In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [16]:
spark = SparkSession.builder.appName("Latihan5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

np.random.seed(7)
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")
print("Siap untuk latihan.")

Siap untuk latihan.


In [17]:
df_solo = df_transaksi.filter(col("kota") == "Solo") \
    .join(df_produk, on="kategori", how="inner") \
    .select("order_id", "kategori", "kota", "pendapatan", "manager")

df_solo.show()

+--------+----------+----+----------+-------+
|order_id|  kategori|kota|pendapatan|manager|
+--------+----------+----+----------+-------+
|     O10|Elektronik|Solo|     72294|   Andi|
|     O12|Elektronik|Solo|     75566|   Andi|
|     O14|Elektronik|Solo|    459719|   Andi|
|     O21|Elektronik|Solo|    176776|   Andi|
|     O22|Elektronik|Solo|     50323|   Andi|
|     O24|Elektronik|Solo|    349635|   Andi|
|     O32|Elektronik|Solo|    464460|   Andi|
|     O44|Elektronik|Solo|    232991|   Andi|
|     O46|Elektronik|Solo|    226277|   Andi|
|     O70|Elektronik|Solo|    229714|   Andi|
|    O103|Elektronik|Solo|    226138|   Andi|
|    O107|Elektronik|Solo|    142577|   Andi|
|    O118|Elektronik|Solo|    375622|   Andi|
|    O130|Elektronik|Solo|    434120|   Andi|
|    O142|Elektronik|Solo|    150915|   Andi|
|    O151|Elektronik|Solo|    212419|   Andi|
|    O173|Elektronik|Solo|    338684|   Andi|
|    O185|Elektronik|Solo|    283907|   Andi|
|    O186|Elektronik|Solo|    4439

In [18]:
window_kota = Window.partitionBy("kota").orderBy(col("pendapatan").desc())
df_top_kota = df_transaksi.withColumn("urutan", row_number().over(window_kota))

df_top_kota.filter(col("urutan") == 1).show()

+--------+------------+--------+----------+------+
|order_id|    kategori|    kota|pendapatan|urutan|
+--------+------------+--------+----------+------+
|     O96|     Fashion|Magelang|    498770|     1|
|    O147|Rumah Tangga|Semarang|    497370|     1|
|    O286|Rumah Tangga|    Solo|    497826|     1|
+--------+------------+--------+----------+------+



In [19]:
hasil_avg_kota = spark.sql('''
    SELECT kota, AVG(pendapatan) AS rata_rata_pendapatan
    FROM transaksi
    GROUP BY kota
    ORDER BY rata_rata_pendapatan DESC
''')
hasil_avg_kota.show()

+--------+--------------------+
|    kota|rata_rata_pendapatan|
+--------+--------------------+
|    Solo|  278926.25925925927|
|Semarang|            267303.8|
|Magelang|   263705.6568627451|
+--------+--------------------+

